# Web Technology 2 — Kivy front end with ATLAS Go: 16 lessons, 11 labs

Companion to the [Web Technology 2 course](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md). ATLAS Go's front end is Python and **Kivy**, not HTML/CSS/JavaScript. Each lesson has a **Try it** program (a complete Kivy app — edit it and re-run) and, for 11 lessons, a **Lab** that measures the lesson's idea with real Kivy and, where possible, the real ATLAS Go widgets, then draws a chart.

Kivy needs a screen, so the setup cell gives this notebook a **virtual display** (Xvfb). Every program opens its window there, checks itself with `assert`, prints a line and closes. Delete the line marked `# auto-close` in a Try-it cell and the program keeps running until you stop the cell.

**Runtime → Run all.** The first run installs Kivy 2.3.1 and Xvfb (about one to two minutes); the lessons then take about two minutes. Nothing here needs an account or real personal data, and nothing contacts a server except GitHub and the package mirrors during setup.

In [ ]:
# Setup: virtual display + Kivy 2.3.1 (the version ATLAS Go ships) + the ATLAS repository.
import os, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Image, display

if shutil.which('xvfb-run') is None:                       # Colab: install Xvfb and Mesa OpenGL
    subprocess.run('apt-get -qq update && apt-get -qq install -y xvfb libgl1-mesa-dri libglx-mesa0 libgl1 > /dev/null',
                   shell=True, check=True)
try:
    import kivy
    assert kivy.__version__ == '2.3.1'
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kivy==2.3.1', 'matplotlib'], check=True)

if Path('kivy_lab/experiments.py').exists():
    ROOT = Path('.').resolve()
elif Path('experiments.py').exists() and Path('../android-app').exists():
    ROOT = Path('..').resolve()
else:
    if not Path('ATLAS').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FUllSCIPTS/ATLAS.git'], check=True)
    ROOT = Path('ATLAS').resolve()
RESULTS = ROOT / 'kivy_lab' / 'results'
ENV = dict(os.environ, KIVY_NO_ARGS='1', KIVY_LOG_MODE='PYTHON', KIVY_NO_CONSOLELOG='1')
NOISE = ('xclip', 'xsel', 'Cutbuffer', 'clipboard')        # harmless: no clipboard tool in the container

def run(script):
    """Run a Kivy program on the virtual display and show what it printed."""
    out = subprocess.run(['xvfb-run', '-a', '-s', '-screen 0 1280x1024x24', sys.executable, script],
                         capture_output=True, text=True, env=ENV, timeout=180)
    for line in out.stdout.splitlines():                   # what the program printed
        if line.strip() and not any(n in line for n in NOISE):
            print(line)
    if out.returncode:                                     # on failure, show the error too
        print(out.stderr[-3000:])
    print('✔ passed' if out.returncode == 0 else f'✘ failed (exit code {out.returncode})')

def lab(n):
    """Run experiment Kn, then show its chart and its observed table."""
    out = subprocess.run(['xvfb-run', '-a', '-s', '-screen 0 1280x1024x24', sys.executable,
                          str(ROOT / 'kivy_lab' / 'experiments.py'), '--experiment', str(n), '--output', str(RESULTS)],
                         capture_output=True, text=True, env=ENV, timeout=300, cwd=ROOT)
    if out.returncode:
        print(out.stderr[-3000:]); raise RuntimeError(f'K{n:02} failed')
    print('\n'.join(l for l in out.stdout.splitlines() if l.startswith('K') or l.startswith('      [')))
    display(Image(filename=str(RESULTS / f'lab_{n:02}.png'), width=760))

print('Kivy ready · repository at', ROOT)

## Lesson 1 — How a Kivy app starts: `App`, `build` and the lifecycle

A Kivy program is a subclass of `kivy.app.App`. You never call your own methods to start it: you call `run()`, and **Kivy calls you back** at fixed moments. `build()` returns the **root widget** — the thing that fills the window. Then Kivy enters its **main loop**: forever, it reads input (touches, keys), runs scheduled callbacks and redraws the screen, about 60 times a second, until the app stops. [Read lesson 1](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-1--how-a-kivy-app-starts-app-build-and-the-lifecycle).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k01.py
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.label import Label

calls = []

class HelloApp(App):
    title = "Hello ATLAS"

    def build(self):                       # 1. make the root widget and return it
        calls.append("build")
        return Label(text="Your city. One app.", font_size=28)

    def on_start(self):                    # 2. the window is up and the root is attached
        calls.append("on_start")
        Clock.schedule_once(self.check, 0.5)

    def on_stop(self):                     # 3. closing: save anything you need here
        calls.append("on_stop")

    def check(self, _dt):
        assert self.root.text == "Your city. One app."   # self.root is what build() returned
        self.stop()                                       # auto-close

HelloApp().run()
print(calls[:2], "…", calls[-1])           # ['build', 'on_start'] … on_stop
assert calls[:2] == ["build", "on_start"] and calls[-1] == "on_stop"


In [ ]:
run('k01.py')

**Lab K01** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(1)

## Lesson 2 — Widgets and the widget tree

Everything you see is a **widget**: `Label`, `Button`, `TextInput`, and containers such as `BoxLayout`. Widgets form a **tree**: `parent.add_widget(child)` hangs a child under a parent; `remove_widget` and `clear_widgets` take them off. Kivy draws the tree from the root down and sends touches from the top down (Lesson 11). Every widget has `parent` and `children`; `widget.walk()` visits the whole subtree. [Read lesson 2](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-2--widgets-and-the-widget-tree).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k02.py
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.label import Label

class TreeApp(App):
    def build(self):
        root = BoxLayout(orientation="vertical")
        root.add_widget(Label(text="ATLAS Go"))
        for name in ("Book a ride", "Order food", "Live map"):
            root.add_widget(Button(text=name))
        return root

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        root = self.root
        names = [w.text for w in reversed(root.children)]     # children are stored newest first!
        assert names == ["ATLAS Go", "Book a ride", "Order food", "Live map"]
        assert all(w.parent is root for w in root.children)
        print(len(list(root.walk())), "widgets:", names)      # 5 widgets: [...]
        root.clear_widgets()                                  # throw the screen away …
        root.add_widget(Label(text="Thanks!"))                # … and build a new one
        assert len(root.children) == 1
        self.stop()                                           # auto-close

TreeApp().run()


In [ ]:
run('k02.py')

**Lab K02** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(2)

## Lesson 3 — Layouts: `BoxLayout`, `size_hint`, padding, spacing and scrolling

A **layout** is a widget that positions and sizes its children. ATLAS Go uses one layout almost everywhere: **`BoxLayout`**, which stacks children in a row (`orientation='horizontal'`, the default) or a column (`'vertical'`). Each child says how big it wants to be: [Read lesson 3](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-3--layouts-boxlayout-size_hint-padding-spacing-and-scrolling).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k03.py
from kivy.app import App
from kivy.clock import Clock
from kivy.metrics import dp
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.scrollview import ScrollView
from kivy.uix.widget import Widget

class LayoutApp(App):
    def build(self):
        root = BoxLayout(orientation="vertical")
        self.header = Widget(size_hint_y=None, height=dp(48))      # fixed
        self.scroll = ScrollView(do_scroll_x=False)                 # hint 1: the rest
        self.footer = Widget(size_hint_y=0.25)                      # a quarter of the scroll's share
        self.body = BoxLayout(orientation="vertical", size_hint_y=None,
                              padding=dp(21), spacing=dp(12))
        self.body.bind(minimum_height=self.body.setter("height"))  # grow with the content
        for i in range(20):
            self.body.add_widget(Button(text=f"Ride {i + 1}", size_hint_y=None, height=dp(48)))
        self.scroll.add_widget(self.body)
        for w in (self.header, self.scroll, self.footer):
            root.add_widget(w)
        return root

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        free = self.root.height - dp(48)
        assert abs(self.header.height - dp(48)) < 1
        assert abs(self.scroll.height - free / 1.25) < 1 and abs(self.footer.height - free / 5) < 1
        expected = 20 * dp(48) + 19 * dp(12) + 2 * dp(21)          # buttons + gaps + padding
        assert abs(self.body.height - expected) < 1
        print("body is", round(self.body.height), "px tall inside a",
              round(self.scroll.height), "px scroll view")
        self.stop()                                                  # auto-close

LayoutApp().run()


In [ ]:
run('k03.py')

**Lab K03** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(3)

## Lesson 4 — Units, colour and screen density: `dp`, `sp` and hex colours

Phones have very different **pixel densities**. A 48-pixel button is a comfortable finger target on an old phone and a tiny sliver on a modern one. Kivy's `kivy.metrics` gives density-independent units: [Read lesson 4](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-4--units-colour-and-screen-density-dp-sp-and-hex-colours).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k04.py
from kivy.app import App
from kivy.clock import Clock
from kivy.core.window import Window
from kivy.metrics import Metrics, dp, sp
from kivy.uix.button import Button
from kivy.utils import get_color_from_hex as color

GREEN = color("#075E4A")
LIME = color("#DFFF73")

class UnitsApp(App):
    def build(self):
        Window.clearcolor = color("#F6F7F2")                   # the window's background
        return Button(text="Book", size_hint=(None, None), size=(dp(160), dp(48)),
                      font_size=sp(14), background_normal="",
                      background_color=GREEN, color=LIME)

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        assert [round(c, 3) for c in GREEN] == [0.027, 0.369, 0.29, 1.0]
        print("density", Metrics.density, "→ dp(48) =", dp(48), "px; sp(14) =", sp(14), "px")
        assert dp(48) == 48 * Metrics.density
        self.stop()                                             # auto-close

UnitsApp().run()


In [ ]:
run('k04.py')

**Lab K04** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(4)

## Lesson 5 — Properties and `bind`: widgets that react

A widget's attributes — `text`, `pos`, `size`, `width`, `height`, `texture_size` … — are **Kivy properties**. Unlike plain Python attributes, a property **announces every change**. `widget.bind(width=callback)` asks Kivy to call `callback(widget, new_value)` whenever `width` changes. Layout itself works this way: a parent changes a child's `size`, and everything bound to it follows. [Read lesson 5](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-5--properties-and-bind-widgets-that-react).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k05.py
from kivy.app import App
from kivy.clock import Clock
from kivy.metrics import dp, sp
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.label import Label
from kivy.uix.widget import Widget

TEXT = ("Passengers book rides and order food. Drivers go online and accept trips. "
        "Food sellers list their kitchen and menu.")

def wrapping_label(text):
    item = Label(text=text, font_size=sp(15), color=(0.09, 0.18, 0.15, 1),
                 size_hint_y=None, halign="left", valign="middle")
    item.bind(width=lambda widget, value: setattr(widget, "text_size", (value, None)))
    item.bind(texture_size=lambda widget, value: setattr(widget, "height", value[1] + dp(3)))
    return item

class WrapApp(App):
    def build(self):
        root = BoxLayout(orientation="vertical", padding=dp(21))
        self.label = wrapping_label(TEXT)
        root.add_widget(self.label)
        root.add_widget(Widget())                       # filler takes the rest
        return root

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        lines = self.label.texture_size[1] / self.label.font_size
        assert self.label.texture_size[0] <= self.label.width + 1    # the text fits the column
        assert lines > 1.5                                           # … because it wrapped
        print(f"wrapped into about {lines:.0f} lines, label is {self.label.height:.0f} px tall")
        self.stop()                                                  # auto-close

WrapApp().run()


In [ ]:
run('k05.py')

**Lab K05** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(5)

## Lesson 6 — Events, callbacks and the Android Back key

Users trigger **events**: a `Button` fires `on_press` when touched and `on_release` when the finger lifts (the usual one to use). You attach a function with `bind(on_release=callback)`; Kivy calls `callback(button)`. Keyboard keys arrive on the **window**: `Window.bind(on_keyboard=handler)`. On Android the **Back** key/gesture arrives as key code **27** (Escape) — return `True` if you handled it, or Kivy closes the app. [Read lesson 6](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-6--events-callbacks-and-the-android-back-key).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k06.py
from kivy.app import App
from kivy.clock import Clock
from kivy.core.window import Window
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.label import Label

SCREENS = ("home", "map", "food", "profile")

class NavApp(App):
    def build(self):
        self.screen = "home"
        root = BoxLayout(orientation="vertical")
        self.title_label = Label(text="home")
        nav = BoxLayout(size_hint_y=0.2)
        self.buttons = {}
        for key in SCREENS:
            button = Button(text=key.title())
            button.bind(on_release=lambda _button, key=key: self.go(key))   # key=key: capture now
            self.buttons[key] = button
            nav.add_widget(button)
        root.add_widget(self.title_label)
        root.add_widget(nav)
        Window.bind(on_keyboard=self.on_keyboard)
        return root

    def go(self, screen):
        self.screen = screen
        self.title_label.text = screen

    def on_keyboard(self, _window, key, *_args):
        if key == 27 and self.screen != "home":    # Back: go home instead of quitting
            self.go("home")
            return True                            # handled — do not close the app
        return False

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        self.buttons["food"].trigger_action(duration=0)          # press it in code
        assert self.title_label.text == "food"
        Window.dispatch("on_keyboard", 27, 0, None, [])          # the Back key
        assert self.screen == "home"
        print("Food button and Back key both work")
        self.stop()                                              # auto-close

NavApp().run()


In [ ]:
run('k06.py')

**Lab K06** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(6)

## Lesson 7 — Text input, choices and dialogs: `TextInput`, `Spinner`, `Popup`

Forms in Kivy are made from three widgets: [Read lesson 7](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-7--text-input-choices-and-dialogs-textinput-spinner-popup).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k07.py
from kivy.app import App
from kivy.clock import Clock
from kivy.metrics import dp
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.label import Label
from kivy.uix.popup import Popup
from kivy.uix.spinner import Spinner
from kivy.uix.textinput import TextInput

def parse_seats(text):
    if not text.strip().isdigit():
        raise ValueError("Seats must be a whole number, for example 2.")
    seats = int(text)
    if not 1 <= seats <= 3:
        raise ValueError("Choose 1 to 3 seats.")
    return seats

class FormApp(App):
    def build(self):
        self.form = {}
        root = BoxLayout(orientation="vertical", padding=dp(16), spacing=dp(8))
        self.seats = TextInput(hint_text="Seats", multiline=False, input_type="number",
                               size_hint_y=None, height=dp(48))
        self.seats.bind(text=lambda _w, value: self.form.__setitem__("seats", value))
        self.place = Spinner(text="UCC", values=("UCC", "Airport", "Kent Station"),
                             size_hint_y=None, height=dp(48))
        self.place.bind(text=lambda _w, value: self.form.__setitem__("place", value))
        book = Button(text="Book", size_hint_y=None, height=dp(48))
        book.bind(on_release=lambda _w: self.safe(self.book))
        for w in (self.seats, self.place, book):
            root.add_widget(w)
        self.popup = None
        return root

    def safe(self, callback):                       # one place turns errors into dialogs
        try:
            callback()
        except ValueError as error:
            self.message("Check your details", str(error))

    def message(self, title, text):
        self.popup = Popup(title=title, content=Label(text=text), size_hint=(0.9, None), height=dp(200))
        self.popup.open()

    def book(self):
        seats = parse_seats(self.form.get("seats", ""))
        self.message("Booked", f"{seats} seat(s) from {self.form.get('place', 'UCC')}")

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        self.seats.text = "lots"                    # as if typed
        self.safe(self.book)
        assert self.popup.title == "Check your details"
        self.popup.dismiss()
        self.seats.text, self.place.text = "2", "Airport"
        self.safe(self.book)
        assert self.popup.content.text == "2 seat(s) from Airport"
        print("bad input → friendly dialog; good input →", self.popup.content.text)
        self.popup.dismiss()
        self.stop()                                 # auto-close

FormApp().run()


In [ ]:
run('k07.py')

## Lesson 8 — Drawing on the canvas: instructions, `canvas.before` and redrawing

Kivy draws with **OpenGL**. Each widget owns a **canvas**: a list of **graphics instructions** run every frame. *Context* instructions change the state (`Color`); *vertex* instructions draw with it (`Rectangle`, `RoundedRectangle`, `Ellipse`, `Line`). There are three canvases per widget, drawn in this order: `canvas.before` → `canvas` → children → `canvas.after`. A background therefore goes in `canvas.before`. [Read lesson 8](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-8--drawing-on-the-canvas-instructions-canvasbefore-and-redrawing).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k08.py
from kivy.app import App
from kivy.clock import Clock
from kivy.graphics import Color, Ellipse, Line, RoundedRectangle
from kivy.metrics import dp
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.widget import Widget
from kivy.utils import get_color_from_hex as color

class Card(BoxLayout):
    """Pattern 1: keep the instruction and update it (ATLAS Go's Panel)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        with self.canvas.before:
            Color(*color("#075E4A"))
            self.background = RoundedRectangle(pos=self.pos, size=self.size, radius=[dp(20)])
        self.bind(pos=self.follow, size=self.follow)

    def follow(self, *_):
        self.background.pos, self.background.size = self.pos, self.size

class Route(Widget):
    """Pattern 2: clear and redraw everything (ATLAS Go's RouteMap)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.bind(pos=self.draw, size=self.draw)

    def draw(self, *_):
        self.canvas.clear()                           # forget the last frame's drawing
        x, y, w, h = self.x, self.y, self.width, self.height
        with self.canvas:
            Color(*color("#DFFF73"))
            Line(points=[x + w * .2, y + h * .2, x + w * .5, y + h * .3, x + w * .8, y + h * .7], width=dp(3))
            for px, py in ((.2, .2), (.8, .7)):
                Ellipse(pos=(x + w * px - dp(9), y + h * py - dp(9)), size=(dp(18), dp(18)))

class CanvasApp(App):
    def build(self):
        self.card = Card(padding=dp(17))
        self.route = Route()
        self.card.add_widget(self.route)
        return self.card

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        before = len(self.route.canvas.children)
        self.route.draw()
        self.route.draw()
        assert len(self.route.canvas.children) == before            # clear() keeps it constant
        assert list(self.card.background.size) == list(self.card.size)
        print("route canvas holds", before, "instructions after every redraw")
        self.stop()                                                  # auto-close

CanvasApp().run()


In [ ]:
run('k08.py')

**Lab K07** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(7)

## Lesson 9 — Text as textures, and charts drawn by hand

On the canvas there is no "text" instruction. Text is first rendered into a **texture** (an image in GPU memory) by `kivy.core.text.Label` (usually imported as `CoreLabel`), then drawn with a `Rectangle(texture=…)`. Rendering text is the expensive part, so code that draws the same words many times should **cache** the textures. [Read lesson 9](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-9--text-as-textures-and-charts-drawn-by-hand).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k09.py
from kivy.app import App
from kivy.clock import Clock
from kivy.core.text import Label as CoreLabel
from kivy.graphics import Color, Rectangle
from kivy.metrics import dp, sp
from kivy.uix.widget import Widget

HOURS = [3, 2, 1, 1, 2, 5, 9, 14, 12, 8, 7, 8, 10, 9, 8, 9, 12, 15, 13, 10, 8, 7, 5, 4]
_cache = {}

def text_texture(text, size=9):
    if text not in _cache:                         # render each distinct text once
        label = CoreLabel(text=text, font_size=sp(size), color=(0.36, 0.43, 0.39, 1))
        label.refresh()
        _cache[text] = label.texture
    return _cache[text]

class Chart(Widget):
    def __init__(self, values, **kwargs):
        super().__init__(**kwargs)
        self.values = values
        self.bind(pos=self.draw, size=self.draw)

    def draw(self, *_):
        self.canvas.clear()
        x, y, w, h = self.x, self.y, self.width, self.height
        base, top, slot = y + dp(20), max(self.values), w / len(self.values)
        self.bars = []
        with self.canvas:
            for hour, value in enumerate(self.values):
                Color(*((0.85, 0.51, 0.17, 1) if value >= 12 else (0.03, 0.37, 0.29, 1)))   # surge in orange
                height = (h - dp(28)) * value / top
                self.bars.append(Rectangle(pos=(x + hour * slot + slot * .15, base), size=(slot * .7, height)))
            for hour in (0, 6, 12, 18):
                tex = text_texture(f"{hour:02d}:00")
                Color(1, 1, 1, 1)                  # white = draw the texture in its own colours
                Rectangle(texture=tex, pos=(x + hour * slot, y), size=tex.size)

class ChartApp(App):
    def build(self):
        return Chart(HOURS)

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        tallest = max(self.root.bars, key=lambda r: r.size[1])
        assert self.root.bars.index(tallest) == HOURS.index(max(HOURS))     # 17:00 is the peak
        assert len(_cache) == 4                                            # four axis labels, rendered once
        print("peak bar at hour", self.root.bars.index(tallest), "·", len(_cache), "cached labels")
        self.stop()                                                        # auto-close

ChartApp().run()


In [ ]:
run('k09.py')

**Lab K08** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(8)

## Lesson 10 — The Clock and the main loop

Kivy runs **one main loop on one thread**. If your code takes 2 seconds, the screen freezes for 2 seconds. Never `time.sleep()` or loop forever in a callback; ask the **Clock** to call you later instead: [Read lesson 10](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-10--the-clock-and-the-main-loop).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k10.py
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.label import Label

class ClockApp(App):
    def build(self):
        self.ticks, self.done, self.draws, self._pending = 0, 0, 0, None
        return Label(text="0")

    def on_start(self):
        Clock.schedule_interval(self.tick, 0.1)       # like ATLAS Go's live_tick (every 1 s)
        Clock.schedule_interval(self.train, 0)        # like lab_step: a slice of work per frame
        for _ in range(50):                           # 50 changes in one frame …
            self.request_draw()
        Clock.schedule_once(self.check, 1.0)

    def tick(self, _dt):
        self.ticks += 1
        self.root.text = str(self.ticks)

    def train(self, _dt):
        self.done += 100                              # pretend to train 100 episodes
        if self.done >= 1000:
            return False                              # finished: stop calling me

    def request_draw(self):                           # … but only one draw
        if self._pending is None:
            self._pending = Clock.schedule_once(self.draw, 0)

    def draw(self, _dt):
        self._pending = None
        self.draws += 1

    def check(self, _dt):
        assert 3 <= self.ticks <= 12 and self.done == 1000 and self.draws == 1   # about 9 ticks in 1 s
        print(f"{self.ticks} ticks, training finished at {self.done}, {self.draws} draw for 50 changes")
        self.stop()                                   # auto-close

ClockApp().run()


In [ ]:
run('k10.py')

**Lab K09** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(9)

## Lesson 11 — Touch input: `on_touch_down`, `move`, `up`, grabbing and clipping

Every finger (or mouse click) is a **touch** object. Kivy offers it to widgets **from the top of the tree down**: the root's `on_touch_down` is called, which by default asks each child in turn. A widget that wants the touch returns `True`, and the search stops. **Important:** every widget is offered every touch, even ones outside it — check `self.collide_point(*touch.pos)` yourself. [Read lesson 11](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-11--touch-input-on_touch_down-move-up-grabbing-and-clipping).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k11.py
from kivy.app import App
from kivy.clock import Clock
from kivy.tests.common import UnitTestTouch
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.stencilview import StencilView
from kivy.uix.widget import Widget

class Pan(StencilView):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.offset, self._last = [0, 0], None

    def on_touch_down(self, touch):
        if not self.collide_point(*touch.pos):
            return super().on_touch_down(touch)       # not mine: let others have it
        touch.grab(self)
        self._last = touch.pos
        return True

    def on_touch_move(self, touch):
        if touch.grab_current is not self:
            return super().on_touch_move(touch)
        self.offset[0] += touch.x - self._last[0]
        self.offset[1] += touch.y - self._last[1]
        self._last = touch.pos
        return True

    def on_touch_up(self, touch):
        if touch.grab_current is self:
            touch.ungrab(self)
            return True
        return super().on_touch_up(touch)

class TouchApp(App):
    def build(self):
        root = BoxLayout(orientation="vertical")
        self.pan = Pan()
        root.add_widget(self.pan)                     # top half
        root.add_widget(Widget())                     # bottom half
        return root

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        x, y = self.pan.center
        touch = UnitTestTouch(x, y)                   # a simulated finger
        touch.touch_down()
        for step in range(1, 11):
            touch.touch_move(x + 10 * step, y - 4 * step)
        touch.touch_move(x + 100, 5)                  # leave the widget: still ours (grabbed)
        touch.touch_up()
        assert self.pan.offset == [100, 5 - y]
        outside = UnitTestTouch(x, 5)                 # starts in the bottom half: ignored
        outside.touch_down(); outside.touch_move(x + 50, 5); outside.touch_up()
        assert self.pan.offset == [100, 5 - y]
        print("panned by", [round(v) for v in self.pan.offset])
        self.stop()                                   # auto-close

TouchApp().run()


In [ ]:
run('k11.py')

**Lab K10** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(10)

## Lesson 12 — Threads and the UI thread

Some work cannot be sliced: a network download blocks until the server answers. That work belongs in a **background thread** (`threading.Thread`). But Kivy's widgets and OpenGL textures belong to the **main thread** — touching them from another thread can crash the app or corrupt the screen. The rule: **workers fetch bytes; the main thread builds widgets and textures.** `Clock.schedule_once` is thread-safe, so a worker hands its result over by scheduling a callback. [Read lesson 12](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-12--threads-and-the-ui-thread).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k12.py
import queue
import threading
import time
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.label import Label

class Loader:
    def __init__(self, on_result, workers=2):
        self.jobs, self.on_result = queue.Queue(), on_result
        for _ in range(workers):
            threading.Thread(target=self.work, daemon=True).start()

    def work(self):                                    # runs on a worker thread
        while True:
            key = self.jobs.get()
            time.sleep(0.05)                           # pretend to download a map tile
            data = f"tile {key}".encode()
            Clock.schedule_once(lambda _dt, key=key, data=data: self.on_result(key, data), 0)

class ThreadApp(App):
    def build(self):
        self.main = threading.get_ident()
        self.results = {}
        self.loader = Loader(self.deliver)
        return Label(text="loading…")

    def deliver(self, key, data):                     # runs on the main thread
        assert threading.get_ident() == self.main
        self.results[key] = data
        self.root.text = f"{len(self.results)} tiles"

    def on_start(self):
        for key in range(8):
            self.loader.jobs.put(key)
        Clock.schedule_once(self.check, 1.0)

    def check(self, _dt):
        assert len(self.results) == 8 and self.root.text == "8 tiles"
        print("8 tiles downloaded in the background and delivered on the main thread")
        self.stop()                                    # auto-close

ThreadApp().run()


In [ ]:
run('k12.py')

**Lab K11** — predict the chart, then run. The [lab manual](https://github.com/FUllSCIPTS/ATLAS/blob/main/kivy_lab/README.md) describes what it measures.

In [ ]:
lab(11)

## Lesson 13 — Screens as state: one `render()` instead of many screens

An app has many screens. Kivy offers two styles: [Read lesson 13](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-13--screens-as-state-one-render-instead-of-many-screens).

**Try it** — example 1 of 2 from the lesson, saved as a file and run on the virtual display:

In [ ]:
%%writefile k13.py
import json
import tempfile
from pathlib import Path
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.label import Label

STATE = Path(tempfile.mkdtemp()) / "ui.json"

class MiniAtlas(App):
    def build(self):
        saved = json.loads(STATE.read_text()) if STATE.exists() else {}
        self.screen = saved.get("screen", "home")        # reopen where the user left
        self.basket = saved.get("basket", 0)
        self.container = BoxLayout(orientation="vertical")
        self.render()
        return self.container

    def go(self, screen):
        self.screen = screen
        STATE.write_text(json.dumps({"screen": self.screen, "basket": self.basket}))
        self.render()

    def render(self):
        self.container.clear_widgets()
        {"home": self.home, "food": self.food}.get(self.screen, self.home)()
        nav = BoxLayout(size_hint_y=0.2)
        for key in ("home", "food"):
            button = Button(text=key.title())
            button.bind(on_release=lambda _w, key=key: self.go(key))
            nav.add_widget(button)
        self.container.add_widget(nav)

    def home(self):
        self.container.add_widget(Label(text="Your city. One app."))

    def food(self):
        self.container.add_widget(Label(text=f"Basket: {self.basket} dishes"))
        add = Button(text="Add a dish")
        add.bind(on_release=lambda _w: self.add_dish())
        self.container.add_widget(add)

    def add_dish(self):
        self.basket += 1
        self.go("food")                                  # state changed → render again

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        self.go("food")
        self.add_dish(); self.add_dish()
        shown = [w.text for w in self.container.walk() if isinstance(w, Label) and "Basket" in w.text]
        assert shown == ["Basket: 2 dishes"]             # the screen always matches the state
        assert json.loads(STATE.read_text()) == {"screen": "food", "basket": 2}
        print(shown[0], "· saved", STATE.read_text())
        self.stop()                                      # auto-close

MiniAtlas().run()


In [ ]:
run('k13.py')

**Try it** — example 2 of 2 from the lesson, saved as a file and run on the virtual display:

In [ ]:
%%writefile k13a.py
from kivy.app import App
from kivy.clock import Clock
from kivy.lang import Builder

KV = """
BoxLayout:
    orientation: "vertical"
    Label:
        id: title
        text: "home"
    BoxLayout:
        size_hint_y: 0.2
        Button:
            text: "Home"
            on_release: title.text = "home"
        Button:
            text: "Food"
            on_release: title.text = "food"
"""

class KvApp(App):
    def build(self):
        return Builder.load_string(KV)

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        food = [b for b in self.root.walk() if getattr(b, "text", "") == "Food"][0]
        food.trigger_action(duration=0)
        assert self.root.ids.title.text == "food"
        print("KV rule bound on_release to", self.root.ids.title.text)
        self.stop()                                      # auto-close

KvApp().run()


In [ ]:
run('k13a.py')

## Lesson 14 — Talking to Android with pyjnius

Kivy runs on Android inside a Java activity. When you need something only Android provides — GPS, permissions, the status-bar size — **pyjnius** lets Python call Java directly: [Read lesson 14](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-14--talking-to-android-with-pyjnius).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k14.py
import os
from kivy.app import App
from kivy.clock import Clock
from kivy.uix.label import Label
from kivy.utils import platform

def android_sdk():
    """Android's API level via pyjnius; None on a desktop."""
    if platform != "android":
        return None
    from jnius import autoclass                     # only importable on Android
    return int(autoclass("android.os.Build$VERSION").SDK_INT)

def location():
    """A real fix needs Android; a desktop can fake one (ATLAS_FAKE_GPS='51.8935,-8.4921')."""
    fake = os.environ.get("ATLAS_FAKE_GPS")
    if platform == "android" or not fake:
        return None
    lat, lon = (float(v) for v in fake.split(","))
    return lat, lon

class AndroidApp(App):
    def build(self):
        return Label(text=f"platform: {platform}")

    def on_start(self):
        Clock.schedule_once(self.check, 0.5)

    def check(self, _dt):
        sdk = android_sdk()
        os.environ["ATLAS_FAKE_GPS"] = "51.8935,-8.4921"
        point = location()
        if platform == "android":
            assert sdk >= 26                         # ATLAS Go's android.minapi
        else:
            assert sdk is None and point == (51.8935, -8.4921)
        print("platform", platform, "· SDK", sdk, "· location", point)
        self.stop()                                  # auto-close

AndroidApp().run()


In [ ]:
run('k14.py')

## Lesson 15 — Packaging for Android with Buildozer

A phone cannot run `python main.py`. **python-for-android** (p4a) compiles Python, Kivy and your dependencies for ARM processors and wraps them in a Java activity; **Buildozer** drives p4a, the Android SDK and NDK from one settings file, `buildozer.spec` (INI format: `key = value` lines in sections). [Read lesson 15](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-15--packaging-for-android-with-buildozer).

## Lesson 16 — Testing a Kivy front end

UI code is hard to test, so **keep as little logic in it as possible**. ATLAS Go's layering makes that possible: [Read lesson 16](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY_2.md#lesson-16--testing-a-kivy-front-end).

**Try it** — the lesson's worked example, saved as a file and run on the virtual display:

In [ ]:
%%writefile k16.py
import os
import tempfile
from pathlib import Path
from kivy.app import App
from kivy.clock import Clock
from kivy.core.window import Window
from kivy.metrics import dp
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.textinput import TextInput

class Screen(App):
    def build(self):
        self.controls, self.signed_in = {}, False
        root = BoxLayout(orientation="vertical", padding=dp(20), spacing=dp(10))
        email = TextInput(hint_text="Email", multiline=False, size_hint_y=None, height=dp(48))
        submit = Button(text="Sign in", size_hint_y=None, height=dp(48))
        submit.bind(on_release=lambda _w: self.sign_in())
        self.controls.update(login_email=email, login_submit=submit)    # stable keys for tests
        root.add_widget(email); root.add_widget(submit)
        return root

    def sign_in(self):
        self.signed_in = "@" in self.controls["login_email"].text

    def on_start(self):
        Clock.schedule_once(self.acceptance, 0.5)

    def acceptance(self, _dt):                      # an end-to-end check, like acceptance.py
        checks = 0
        def check(condition, message):
            nonlocal checks
            assert condition, message
            checks += 1
        check(all(w.width >= dp(40) and w.height >= dp(44) for w in self.controls.values()),
              "controls are big enough to touch")
        self.controls["login_email"].text = "ana@example.ie"
        self.controls["login_submit"].trigger_action(duration=0)
        check(self.signed_in, "signed in after pressing the button")
        shot = Path(tempfile.mkdtemp()) / "login.png"
        Path(Window.screenshot(name=str(shot))).replace(shot)       # Kivy adds a number; rename it
        check(shot.exists() and shot.stat().st_size > 0, "screenshot saved")
        print(checks, "checks passed; screenshot", shot.name)
        self.stop()                                 # auto-close

Screen().run()


In [ ]:
run('k16.py')

## All labs as a self-test, and your results

Runs every lab's assertions in one go, then packs the charts, CSV and JSON files into `kivy_11_labs.zip` (download it from the **Files** panel on the left).

In [ ]:
out = subprocess.run(['xvfb-run', '-a', '-s', '-screen 0 1280x1024x24', sys.executable, 'kivy_lab/experiments.py'],
                     capture_output=True, text=True, env=ENV, cwd=ROOT, timeout=900)
print([l for l in out.stdout.splitlines() if 'labs ran' in l][0] if out.returncode == 0 else out.stderr[-3000:])
shutil.make_archive('kivy_11_labs', 'zip', RESULTS)
print('saved kivy_11_labs.zip with', len(list(RESULTS.glob('lab_*'))), 'files')